# 04 — HANS prediction validation (updated)

**Purpose:** verify whether the optimized **`int8_compiled_fp32_head`** variant preserves the behavior of the original TorchAO INT8 model. This notebook does not retrain, edit, or save a model checkpoint.

Unlike the original notebook, this version:

- Recognizes the new compiled INT8 variant.
- Compares it to **both** `int8_eager` (CUDA) **and** the original saved `int8_seed{seed}_predictions.csv`.
- Uses `pairID` to align the complete HANS set, including final-batch duplicate removal performed by the corrected `worker.py`.
- Reports MNLI 3-class prediction flips, HANS binary prediction flips, and probability changes.
- Only automatically attempts compiled variants with a **successful benchmark status**.

**Important:** the new compiled INT8 model keeps the three-class classification head in FP32. It is *not* identical to the original full-W8A8 model. Similar behavior must be established from results, not assumed.

In [1]:
from pathlib import Path
import subprocess
import sys
import json
import pandas as pd

def find_analysis_folder():
    here = Path.cwd().resolve()
    for directory in [here, *here.parents]:
        for candidate in (directory / 'notebooks/runtime_analysis', directory / 'runtime_analysis', directory):
            candidate = candidate.resolve()
            if (candidate / 'config.json').is_file() and (candidate / 'worker.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate notebooks/runtime_analysis. Check notebook location.')

ANALYSIS_DIR = find_analysis_folder()
REPO_ROOT = ANALYSIS_DIR.parent.parent
CONFIG = json.loads((ANALYSIS_DIR / 'config.json').read_text(encoding='utf-8'))
RESULTS_ROOT = REPO_ROOT / CONFIG['results_dir']
print('Analysis directory:', ANALYSIS_DIR)
print('Project root:', REPO_ROOT)
print('Results directory:', RESULTS_ROOT)

Analysis directory: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis
Project root: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers
Results directory: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis


## 🔴 User settings — run seed 42 first

Change `SEEDS` only after seed 42 is validated. Eager variants are always included to establish a current CUDA reference. Compiled variants are included when a successful benchmark is recorded. The known-failing original `int8_compiled` variant is deliberately excluded from automatic selection.

If your `03_compiled_experiment.ipynb` succeeded for the new INT8 FP32-head variant, it should be included automatically.

In [2]:
SEEDS = [42]  # 🔴 Change to [42, 123, 456] after validating seed 42.
RUN_PREDICTIONS = True  # Set False ONLY if the variant predictions.csv files already exist.

BASE_VARIANTS = ['fp32_eager', 'int8_eager']
OPTIONAL_COMPILED = ['fp32_compiled', 'int8_compiled_fp32_head']

def is_successful_benchmark(seed, variant):
    folder = RESULTS_ROOT / f'seed{seed}' / variant
    status_path = folder / 'benchmark_status.json'
    summary_path = folder / 'benchmark_summary.json'
    if not (status_path.is_file() and summary_path.is_file()):
        return False
    status = json.loads(status_path.read_text(encoding='utf-8'))
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    return (
        status.get('status') == 'success'
        and summary.get('status') == 'success'
        and summary.get('variant') == variant
        and int(summary.get('seed', -1)) == seed
    )

jobs = {}
for seed in SEEDS:
    variants = BASE_VARIANTS.copy()
    variants.extend(v for v in OPTIONAL_COMPILED if is_successful_benchmark(seed, v))
    jobs[seed] = variants
    print(f'Seed {seed}: {variants}')
    if 'int8_compiled_fp32_head' not in variants:
        print('  Note: compiled INT8 FP32-head benchmark not found as successful; this comparison will not run yet.')

Seed 42: ['fp32_eager', 'int8_eager', 'fp32_compiled', 'int8_compiled_fp32_head']


## 1 — Generate HANS prediction CSVs (fresh isolated processes)

The corrected `worker.py` pads only the final incomplete physical batch and **discards predictions for duplicated inputs**. Every output CSV should contain exactly 30,000 real HANS `pairID` rows with no duplicates.

If one compiled variant fails here, the notebook continues and reports the failure. This doesn't establish prediction equivalence: only a successfully generated and compared prediction CSV does.

In [3]:
if RUN_PREDICTIONS:
    failed_jobs = []
    for seed, variants in jobs.items():
        cmd = [
            sys.executable, str(ANALYSIS_DIR / 'run_experiments.py'),
            '--seeds', str(seed), '--variants', *variants,
            '--task', 'predictions', '--continue-on-error',
        ]
        print('\nLaunching:', ' '.join(cmd), flush=True)
        completed = subprocess.run(cmd, cwd=REPO_ROOT, check=False)
        if completed.returncode != 0:
            failed_jobs.append(seed)
            print(f'Seed {seed}: one or more variants failed. See their predictions_status.json files.')
    if failed_jobs:
        print('Failed seeds:', failed_jobs)
else:
    print('Skipping generation; will compare existing predictions.csv files.')


Launching: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/bin/python /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis/run_experiments.py --seeds 42 --variants fp32_eager int8_eager fp32_compiled int8_compiled_fp32_head --task predictions --continue-on-error
CUDA variants run SEQUENTIALLY in a FRESH PROCESS each.
Working directory: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers

Running seed=42, variant=fp32_eager, task=predictions



W1025 23:08:49.271000 48780 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 23:08:49.287000 48780 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21289.81it/s]


Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/fp32_eager/predictions.csv
Finished exit code 0 in 15.3s

Running seed=42, variant=int8_eager, task=predictions



W1025 23:09:04.534000 48870 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 23:09:04.550000 48870 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11180.51it/s]


Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/int8_eager/predictions.csv
Finished exit code 0 in 56.3s

Running seed=42, variant=fp32_compiled, task=predictions



W1025 23:10:00.925000 49054 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 23:10:00.941000 49054 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19465.15it/s]


Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/fp32_compiled/predictions.csv
Finished exit code 0 in 23.4s

Running seed=42, variant=int8_compiled_fp32_head, task=predictions



W1025 23:10:24.307000 49261 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 23:10:24.323000 49261 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11574.54it/s]
/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/lib/python3.12/site-packages/torch/_functorch/_aot_autograd/autograd_cache.py:707: UserWarning: Int8Tensor does not implement _stable_hash_for_caching. For PT2-compatible tensor subclasses, it is recommended to implement _stable_hash_for_caching(self) -> str for stable AOT autograd cach

Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/int8_compiled_fp32_head/predictions.csv
Finished exit code 0 in 31.7s

All requested subprocess jobs completed successfully.


## 2 — Compare with the original and current eager predictions

`compare_predictions.py` makes six possible checks (skipping missing files):

- Existing saved FP32 HANS vs FP32 eager CUDA.
- Existing saved INT8 HANS vs INT8 eager CUDA.
- FP32 compiled vs FP32 eager CUDA.
- Original full-INT8 compiled vs eager, if available.
- **INT8 compiled FP32-head vs INT8 eager CUDA.**
- **INT8 compiled FP32-head vs the original saved INT8 HANS predictions.**

The reference used to produce your original HANS behavioral findings may have been CPU, so some differences between those and new CUDA predictions are possible. Report them instead of silently treating them as the same model execution.

In [4]:
cmd = [
    sys.executable, str(ANALYSIS_DIR / 'compare_predictions.py'),
    '--seeds', *map(str, SEEDS),
]
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

status_csv = RESULTS_ROOT / 'prediction_equivalence_status.csv'
checks_csv = RESULTS_ROOT / 'prediction_equivalence_checks.csv'
print('Comparison availability:')
display(pd.read_csv(status_csv))
if checks_csv.exists():
    checks = pd.read_csv(checks_csv)
    selected_columns = [
        'seed', 'reference_variant', 'candidate_variant', 'examples',
        'mnli_3way_disagreement_n', 'hans_binary_disagreement_n',
        'probability_mae', 'probability_max_abs_difference',
        'exact_label_agreement', 'all_probabilities_within_tolerance',
    ]
    display(checks[[c for c in selected_columns if c in checks.columns]])
else:
    print('No complete comparisons yet; check missing prediction files or errors above.')

OK seed 42: Original FP32 reference vs FP32 eager CUDA | MNLI flips=0 | HANS flips=0 | P-MAE=1.55975e-07
OK seed 42: Original INT8 reference vs INT8 eager CUDA | MNLI flips=383 | HANS flips=375 | P-MAE=0.00823499
OK seed 42: FP32 compiled vs FP32 eager CUDA | MNLI flips=0 | HANS flips=0 | P-MAE=8.64888e-08
SKIP seed 42: Original INT8 compiled vs INT8 eager CUDA (missing prediction file)
OK seed 42: INT8 compiled FP32 head vs INT8 eager CUDA | MNLI flips=367 | HANS flips=356 | P-MAE=0.00810566
OK seed 42: INT8 compiled FP32 head vs original INT8 reference | MNLI flips=376 | HANS flips=369 | P-MAE=0.00818973
Saved comparisons: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/prediction_equivalence_checks.csv
Saved comparison availability: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/prediction_equivalence_status.csv
A compiled FP32-head INT8 prediction match must be measured;

,seed,comparison,reference_file,candidate_file,status,reason,details_file
0,42,Original FP32 reference vs FP32 eager CUDA,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,completed,NaN,/home/mohnishr/workspace/DLBA/Quantization-Spu...
1,42,Original INT8 reference vs INT8 eager CUDA,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,completed,NaN,/home/mohnishr/workspace/DLBA/Quantization-Spu...
2,42,FP32 compiled vs FP32 eager CUDA,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,completed,NaN,/home/mohnishr/workspace/DLBA/Quantization-Spu...
3,42,Original INT8 compiled vs INT8 eager CUDA,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,skipped,Missing reference and/or candidate predictions,NaN
4,42,INT8 compiled FP32 head vs INT8 eager CUDA,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,completed,NaN,/home/mohnishr/workspace/DLBA/Quantization-Spu...
5,42,INT8 compiled FP32 head vs original INT8 refer...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,completed,NaN,/home/mohnishr/workspace/DLBA/Quantization-Spu...


,seed,reference_variant,candidate_variant,examples,mnli_3way_disagreement_n,hans_binary_disagreement_n,probability_mae,probability_max_abs_difference,exact_label_agreement,all_probabilities_within_tolerance
0,42,original_fp32,fp32_eager,30000,0,0,1.559748e-07,0.000008,True,True
1,42,original_int8,int8_eager,30000,383,375,8.234993e-03,0.365907,False,False
2,42,fp32_eager,fp32_compiled,30000,0,0,8.648885e-08,0.000004,True,True
3,42,int8_eager,int8_compiled_fp32_head,30000,367,356,8.105662e-03,0.291978,False,False
4,42,original_int8,int8_compiled_fp32_head,30000,376,369,8.189728e-03,0.621089,False,False


## 3 — Focus on the new INT8 FP32-head variant

**Use this check before presenting the compiled variant alongside the original HANS behavior results.** Three-class flips can occur within the HANS non-entailment category even when binary HANS labels do not change; therefore report both.

The `prediction_differences/` folder contains example-level changes and probability differences for each completed comparison. A small probability MAE **does not imply** there are zero classification changes near a decision boundary.

In [5]:
if checks_csv.exists():
    checks = pd.read_csv(checks_csv)
    focus = checks[checks['candidate_variant'].eq('int8_compiled_fp32_head')]
    if len(focus):
        display(focus)
        if (focus['hans_binary_disagreement_n'] > 0).any():
            print('WARNING: HANS binary predictions changed; do NOT reuse original INT8 HANS metrics without reevaluation.')
        elif (focus['mnli_3way_disagreement_n'] > 0).any():
            print('MNLI predictions changed even though HANS binary labels may agree. Review probability/3-way differences.')
        else:
            print('No 3-way label flips in the available completed comparisons. Review probability differences too.')
    else:
        print('No completed FP32-head INT8 comparison. Do not make an equivalence claim yet.')
else:
    print('Missing equivalence report. Run the comparison step above.')

,seed,comparison,reference_variant,candidate_variant,reference_file,candidate_file,examples,mnli_3way_disagreement_n,mnli_3way_disagreement_percent,hans_binary_disagreement_n,hans_binary_disagreement_percent,probability_mae,probability_p99_abs_difference,probability_max_abs_difference,fraction_probability_elements_over_tolerance,tolerance,exact_label_agreement,all_probabilities_within_tolerance,num_flagged_examples,flagged_examples_csv
3,42,INT8 compiled FP32 head vs INT8 eager CUDA,int8_eager,int8_compiled_fp32_head,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,30000,367,1.223333,356,1.186667,0.008106,0.099652,0.291978,0.498611,0.001,False,False,18893,/home/mohnishr/workspace/DLBA/Quantization-Spu...
4,42,INT8 compiled FP32 head vs original INT8 refer...,original_int8,int8_compiled_fp32_head,/home/mohnishr/workspace/DLBA/Quantization-Spu...,/home/mohnishr/workspace/DLBA/Quantization-Spu...,30000,376,1.253333,369,1.230000,0.008190,0.099235,0.621089,0.501278,0.001,False,False,18975,/home/mohnishr/workspace/DLBA/Quantization-Spu...


## Next

Once the original-INT8-versus-new-INT8 comparison has been reviewed, open **`05_review_results.ipynb`**. The summary notebook will identify the modified INT8 variant separately and exclude mismatched benchmarking protocols instead of silently combining their numbers.